In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from pathlib import Path

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Load all JSON files from the downloaded folder
data_dir = Path('../data/downloaded')
all_records = []

print("Loading data from multiple files...")

# Walk through all subdirectories and find JSON files
for json_file in data_dir.rglob('*.json'):  # rglob finds files recursively
    print(f"Reading: {json_file}")
    try:
        with open(json_file, 'r') as f:
            file_data = json.load(f)
            
            # Check if it has 'records' key (your processed format)
            if isinstance(file_data, dict) and 'records' in file_data:
                all_records.extend(file_data['records'])
            # Or if it's a list directly
            elif isinstance(file_data, list):
                all_records.extend(file_data)
    except Exception as e:
        print(f"  Error reading {json_file}: {e}")

print(f"\n Loaded {len(all_records)} total records from {len(list(data_dir.rglob('*.json')))} files")

# Convert to DataFrame
df = pd.DataFrame(all_records)

# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Basic info
print("=== DATASET OVERVIEW ===")
print(f"Total records: {len(df)}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Number of days: {(df['timestamp'].max() - df['timestamp'].min()).days}")
print(f"\nCities: {df['city'].unique().tolist()}")
print(f"\nRecords per city:")
print(df['city'].value_counts())

# Check for missing values
print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

# Statistical summary
print("\n=== TEMPERATURE STATISTICS ===")
print(df.groupby('city')['temperature_celsius'].describe())

# Visualize temperature trends
plt.figure(figsize=(14, 8))
for city in df['city'].unique():
    city_data = df[df['city'] == city].sort_values('timestamp')
    plt.plot(city_data['timestamp'], city_data['temperature_celsius'], 
             marker='o', label=city, alpha=0.7)

plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.title('Temperature Trends Over Time by City')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/temp_trends.png', dpi=300)
plt.show()

# Temperature distribution
plt.figure(figsize=(12, 6))
df.boxplot(column='temperature_celsius', by='city', figsize=(12, 6))
plt.suptitle('Temperature Distribution by City')
plt.xlabel('City')
plt.ylabel('Temperature (°C)')
plt.tight_layout()
plt.savefig('../data/temp_distribution.png', dpi=300)
plt.show()

# Correlation heatmap
numeric_cols = ['temperature_celsius', 'feels_like', 'humidity_percent', 
                'pressure_hpa', 'wind_speed_mps', 'cloudiness_percent']
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../data/correlation_heatmap.png', dpi=300)
plt.show()

print("\n Exploration complete.")